In [1]:
#---------------------------------------
# Import Libreries
#---------------------------------------
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\venut\OneDrive\Documents\Projects\credit_risk_project


In [2]:
#---------------------------------------
# 1. Load Required Data only
#---------------------------------------
required_cols = [
    "loan_status", "loan_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "emp_length", "home_ownership",
    "annual_inc", "verification_status", "purpose", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high",
    "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "application_type"
]

# Only keep loans with resolved outcomes to avoid label ambiguity
resolved_statuses = ["Fully Paid", "Charged Off", "Default"]

chunks = []
for chunk in pd.read_csv(
    "data/raw/dataset.csv",
    usecols=required_cols,
    chunksize=100_000,
    low_memory=False
):
    chunk = chunk[chunk["loan_status"].isin(resolved_statuses)]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print("Shape after filtering to resolved loans:", df.shape)

Shape after filtering to resolved loans: (1345350, 22)


In [3]:
#---------------------------------------
# 2. Create Target Variable
#---------------------------------------
df["target"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1
})

print("Target distribution:")
print(df["target"].value_counts())
print(f"Default rate: {df['target'].mean():.2%}")

Target distribution:
target
0    1076751
1     268599
Name: count, dtype: int64
Default rate: 19.96%


In [4]:
#---------------------------------------
# 3. Data Type Corrections
#---------------------------------------

# Interest rate: strip % if stored as string
if df["int_rate"].dtype == object:
    df["int_rate"] = df["int_rate"].astype(str).str.replace("%", "", regex=False)
    df["int_rate"] = pd.to_numeric(df["int_rate"], errors="coerce")

# Revolving utilization: strip % if stored as string
if df["revol_util"].dtype == object:
    df["revol_util"] = df["revol_util"].astype(str).str.replace("%", "", regex=False)
    df["revol_util"] = pd.to_numeric(df["revol_util"], errors="coerce")

print("int_rate dtype:", df["int_rate"].dtype)
print("revol_util dtype:", df["revol_util"].dtype)

int_rate dtype: float64
revol_util dtype: float64


In [5]:
#---------------------------------------
# 4. Feature Engineering
#---------------------------------------

# 1. Loan-to-Income Ratio — captures affordability
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)

# 2. Average FICO Score
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

# 3. Loan Term as Numeric (36 or 60)
df["loan_term_numeric"] = pd.to_numeric(
    df["term"].str.extract(r"(\d+)")[0], errors="coerce"
)

# 4. Installment-to-Income Ratio — monthly payment burden
df["installment_to_income"] = df["installment"] / (df["annual_inc"].replace(0, np.nan) / 12)

# 5. DTI capped at 100 to remove erroneous outliers
df["dti_capped"] = df["dti"].clip(upper=100)

print("Engineered features added:")
print(["loan_to_income", "fico_avg", "loan_term_numeric", "installment_to_income", "dti_capped"])
print(df[["loan_to_income", "fico_avg", "loan_term_numeric", "installment_to_income", "dti_capped"]].describe())

Engineered features added:
['loan_to_income', 'fico_avg', 'loan_term_numeric', 'installment_to_income', 'dti_capped']
       loan_to_income      fico_avg  loan_term_numeric  installment_to_income  \
count    1.344989e+06  1.345350e+06       1.345350e+06           1.344989e+06   
mean     3.973523e-01  6.981853e+02       4.179029e+01           1.423694e-01   
std      7.082765e+01  3.185312e+01       1.026838e+01           2.364355e+01   
min      1.714286e-04  6.270000e+02       3.600000e+01           6.912000e-05   
25%      1.246575e-01  6.720000e+02       3.600000e+01           4.626102e-02   
50%      2.000000e-01  6.920000e+02       3.600000e+01           7.219950e-02   
75%      2.909091e-01  7.120000e+02       3.600000e+01           1.054020e-01   
max      4.000000e+04  8.475000e+02       6.000000e+01           1.320792e+04   

         dti_capped  
count  1.344976e+06  
mean   1.821677e+01  
std    8.828183e+00  
min   -1.000000e+00  
25%    1.179000e+01  
50%    1.761000e+01 

In [6]:
#---------------------------------------
# 5. Missing Value Analysis
#---------------------------------------

missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_df[missing_df["Missing Count"] > 0])

                       Missing Count  Missing %
emp_length                     78516   5.836102
revol_util                       857   0.063701
dti                              374   0.027799
dti_capped                       374   0.027799
loan_to_income                   361   0.026833
installment_to_income            361   0.026833


In [7]:
#---------------------------------------
# 6. Basic Validation Checks
#---------------------------------------

assert df["target"].isin([0, 1]).all(), "Target contains unexpected values"
assert df["loan_amnt"].gt(0).all(), "Loan amount contains non-positive values"
assert df["fico_avg"].between(300, 850).dropna().all(), "FICO avg out of expected range"
assert df["loan_term_numeric"].isin([36, 60]).dropna().all(), "Unexpected term values"

print("All validation checks passed.")

All validation checks passed.


In [8]:
#---------------------------------------
# 7. Save Processed Dataset
#---------------------------------------   

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/processed_data.csv", index=False)
print("Saved processed dataset to data/processed/processed_data.csv")
print("Shape:", df.shape)

Saved processed dataset to data/processed/processed_data.csv
Shape: (1345350, 28)


In [ ]:
#---------------------------------------
# 8. Final Dataset for Modeling
#---------------------------------------

os.makedirs("data/processed", exist_ok=True)
df.to_csv(
    "data/processed/final_model_data.csv",
    index=False
)
print("Processed dataset saved.")

Processed dataset saved.
